# Waste sites: what we tested and what we found

The platform scores one 80 m window of free Esri imagery and says how much it looks like a waste site.

The model has two parts:

1. **ResNet-50** (He et al., 2016), a published image network trained on ImageNet. It is not retrained here. It turns each image into 2,048 numbers.
2. **A logistic regression** fitted on our Guinean labels. This small last step is the only part fitted in Guinea.

This is a known approach for small label sets (Razavian et al., 2014). It is not a new method.

The script outputs are in `2_dumpsites`.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.max_colwidth", 60, "display.width", 180)
ROOT = Path(r"D:\Guinea_Methodology\plasticwaste_experiments")

OUT = ROOT / "2_dumpsites" / "outputs"

## 1. The two kinds of test

- **Labelled points.** Windows centred on 111 known sites and 540 other places. Most of those other places are rural, so this is an easy test.
- **Blind grids.** Every window over Conakry (422) and Boké (557), labelled by eye. This is the honest test for finding sites nobody selected.

In [2]:
pts = pd.read_csv(OUT / "set_points.csv")
print("Labelled points:", len(pts), "sites:", int(pts.y.sum()))
for town in ("conakry", "boke"):
    g = pd.read_csv(OUT / f"set_{town}.csv")
    print(f"{town} grid:", len(g), "windows, share with waste:", round(g.y.mean(), 3))

Labelled points: 651 sites: 111
conakry grid: 422 windows, share with waste: 0.261
boke grid: 557 windows, share with waste: 0.436


## 2. Five ways of describing an image

We kept the logistic step and changed what feeds it. AerialWaste is the only one trained on waste imagery. Colour and texture is a simple control with no deep learning.

AUC is the chance a real site scores above a non-site: 0.5 is a coin toss, 1 is perfect.

In [3]:
b = pd.read_csv(OUT / "02_benchmark_wide.csv", header=[0, 1], index_col=0)
b.columns = [f"{a.split('.')[0]} {c}".strip() for a, c in b.columns]
b = b.drop(index="mean_key", errors="ignore")
b = b[["auc boke", "auc conakry"]].astype(float)
b.index.name = "Model and training labels"
b.sort_values("auc boke", ascending=False).round(3)

,auc boke,auc conakry
Model and training labels,,
ImageNet ResNet-50 (deployed) | frozen deployed head,0.687,0.594
AerialWaste ResNet-50 FPN (waste-trained) | points,0.679,0.590
ConvNeXt-Base (ImageNet) | points,0.664,0.633
AerialWaste ResNet-50 FPN (waste-trained) | both,0.654,0.554
ImageNet ResNet-50 (deployed) | both,0.647,0.584
ImageNet ResNet-50 (deployed) | points,0.644,0.607
ConvNeXt-Base (ImageNet) | both,0.581,0.656
ViT-B/16 SWAG | points,0.554,0.589
Colour and texture summary (control) | points,0.533,0.638


**Reading it.** The platform's model reproduces exactly, 0.594 at Conakry and 0.687 at Boké. No model beats it by more than its interval. Colour and texture is close at Conakry, which is a warning: the score is tracking bare, disturbed ground more than waste.

## 3. Would labels from the same town help?

If the problem were moving between towns, training inside each town would fix it.

In [4]:
pd.read_csv(OUT / "04_within_town.csv").round(3)[["test", "rep", "auc", "auc_p05", "auc_p95"]]

,test,rep,auc,auc_p05,auc_p95
0,conakry,Colour and texture summary (control),0.636,0.558,0.694
1,conakry,ImageNet ResNet-50 (deployed),0.620,0.553,0.674
2,conakry,AerialWaste ResNet-50 FPN (waste-trained),0.633,0.589,0.670
3,conakry,ConvNeXt-Base (ImageNet),0.598,0.512,0.672
4,conakry,ViT-B/16 SWAG,0.613,0.559,0.666
5,boke,Colour and texture summary (control),0.686,0.622,0.747
6,boke,ImageNet ResNet-50 (deployed),0.716,0.655,0.765
7,boke,AerialWaste ResNet-50 FPN (waste-trained),0.722,0.664,0.772
8,boke,ConvNeXt-Base (ImageNet),0.715,0.657,0.763
9,boke,ViT-B/16 SWAG,0.719,0.662,0.767


**Reading it.** Still 0.60-0.64 at Conakry and 0.69-0.72 at Boké, whichever model is used. Better models and local labels do not change much. The limit is what the images show at this scale.

## 4. What we tried before this benchmark

These are from the methodology document.

| Tried | Result | Decision |
|---|---|---|
| Sentinel-2 plastic indices on land | They track bare ground, same as NDVI | Dropped |
| FDI on land | Same bare-ground signal | Dropped |
| 250 m windows | Scored towns above dumps | Moved to 80 m |
| Lab's Guatemala checkpoint | Lowered accuracy on Guinean imagery | Dropped |
| AerialWaste's own model and head | Its last layers did not work here | Used its features in tests only; no gain over ImageNet |
| Detector like Guatemala's | 111 sites are too few to train or validate one | Not attempted |
| ResNet-50 with a logistic step | Works at 80 m, best supported by the data | Kept |

## 5. Why a location check and not a national map

The blind grids give the answer. At the platform's default level:

In [5]:
pd.DataFrame({"Town": ["Conakry", "Boké"], "Precision": [0.347, 0.586], "Share of waste windows anyway": [0.261, 0.436],
              "Wrong flags, about": ["2 in 3", "2 in 5"]})

,Town,Precision,Share of waste windows anyway,"Wrong flags, about"
0,Conakry,0.347,0.261,2 in 3
1,Boké,0.586,0.436,2 in 5


A national map would be mostly false alarms and would look like an inventory. It would also be built on labels from a sample of places, so it may not hold elsewhere.

A pin is different, because a person already suspects the place. On labelled points the model reaches 0.832. In a town it behaves more like the blind grids, so the check should be read as a lead for a field visit.

## 6. A small cost of the check

The check uses a fixed 80 m grid, so a pin is not exactly centred. We measured the cost.

In [6]:
w = json.loads((OUT / "03_location_check.json").read_text())
print(f"Centred AUC {w['centred_auc']:.3f}, fixed grid {w['lattice_auc']:.3f}, median offset {w['pin_offset_median_m']:.0f} m")
print("These two AUCs use one fixed setting, so only their difference matters. The published figure is 0.832.")

Centred AUC 0.900, fixed grid 0.890, median offset 32 m
These two AUCs use one fixed setting, so only their difference matters. The published figure is 0.832.


## What this means

- The method is standard and every step is published.
- It finds disturbed ground where waste often lies. It does not identify waste itself.
- Keep it as a location check with the blind-grid figures beside it.